### 직렬화와 역직렬화로 모델 저장 및 로드하기

In [1]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_teddynote import logging


load_dotenv()
logging.langsmith("test0914")


print("OpenAI 키 로드됨 : ", bool(os.getenv("OPENAI_API_KEY")))
print("LangSmith 키 로드됨 : ", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith 프로젝트 : ", os.getenv("LANGSMITH_PROJECT"))

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914
OpenAI 키 로드됨 :  True
LangSmith 키 로드됨 :  True
LangSmith 프로젝트 :  test0914


In [2]:
prompt = PromptTemplate.from_template("{fruit}의 색상이 무엇입니까?")

In [3]:
print(f"ChatOpenAI : {ChatOpenAI.is_lc_serializable()}") #True로 나오면 직렬화가 가능하다는 의미

ChatOpenAI : True


In [4]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print(f"ChatOpenAI: {llm.is_lc_serializable()}") # llm객체에 대하여 직렬화 가능 여부를 확인

ChatOpenAI: True


In [5]:
chain = prompt | llm

chain.is_lc_serializable()  # chain이 직렬화가 가능한지 확인

True

- 체인 직렬화하기

In [ ]:
from langchain_core.load import dumpd, dumps

dumpd_chain = dumpd(chain)  # 딕셔너리 구조
dumpd_chain

{'lc': 1,
 'type': 'constructor',
 'id': ['langchain', 'schema', 'runnable', 'RunnableSequence'],
 'kwargs': {'first': {'lc': 1,
   'type': 'constructor',
   'id': ['langchain', 'prompts', 'prompt', 'PromptTemplate'],
   'kwargs': {'input_variables': ['fruit'],
    'template': '{fruit}의 색상이 무엇입니까?',
    'template_format': 'f-string'},
   'name': 'PromptTemplate'},
  'last': {'lc': 1,
   'type': 'constructor',
   'id': ['langchain', 'chat_models', 'openai', 'ChatOpenAI'],
   'kwargs': {'model_name': 'gpt-4o-mini',
    'temperature': 0.0,
    'openai_api_key': {'lc': 1, 'type': 'secret', 'id': ['OPENAI_API_KEY']},
    'stream_usage': True},
   'name': 'ChatOpenAI'}},
 'name': 'RunnableSequence'}

In [7]:
dumps_chain = dumps(chain) # 문자열로 직렬화
dumps_chain

'{"lc": 1, "type": "constructor", "id": ["langchain", "schema", "runnable", "RunnableSequence"], "kwargs": {"first": {"lc": 1, "type": "constructor", "id": ["langchain", "prompts", "prompt", "PromptTemplate"], "kwargs": {"input_variables": ["fruit"], "template": "{fruit}\\uc758 \\uc0c9\\uc0c1\\uc774 \\ubb34\\uc5c7\\uc785\\ub2c8\\uae4c?", "template_format": "f-string"}, "name": "PromptTemplate"}, "last": {"lc": 1, "type": "constructor", "id": ["langchain", "chat_models", "openai", "ChatOpenAI"], "kwargs": {"model_name": "gpt-4o-mini", "temperature": 0.0, "openai_api_key": {"lc": 1, "type": "secret", "id": ["OPENAI_API_KEY"]}, "stream_usage": true}, "name": "ChatOpenAI"}}, "name": "RunnableSequence"}'

Pickle 파일로 직렬화하고 로드하기

In [11]:
import pickle

# fruit_chain.pkl 파일로 직렬화된 체인을 저장
with open("fruit_chain.pkl","wb") as f:  # wb는 파일을 쓰기 전용으로 열고 데이터를 바이너리 형식으로 저장
    pickle.dump(dumpd_chain, f)

In [12]:
import json

with open("fruit_chain.json", "w") as fp: # 파일을 쓰기모드로 열어 파일 객체 fp를 생성해 json형식으로 저장
    json.dump(dumpd_chain, fp)

In [13]:
with open("fruit_chain.pkl", "rb") as f: # rb 읽기모드
    loaded_chain = pickle.load(f) # load 메서드로 f에 저장된 직렬화된 데이터를 역직렬화하여 loaded_chain객체에 복원

In [16]:
from langchain_core.load import load

# load()는 JSON파이을 읽어 파이썬 객체로 되살리는 함수
# 아무 클래스나 복원하면 악의적인 파일로 임의 코드가 실행될 수 있음
# 기본값이 langchain-core 내부 클래스만 허용하도록 잠겨 있음
# ChatOpenAI는 langchain-openai 라는 별도 패키지 소속이라 막혀있음 
chain_from_file = load(loaded_chain, allowed_objects="all")

print(chain_from_file.invoke({"fruit":"사과"}))

content='사과의 색상은 다양합니다. 일반적으로 빨간색, 초록색, 노란색 등 여러 가지 색상이 있으며, 품종에 따라 다르게 나타납니다. 예를 들어, 레드 딜리셔스는 주로 빨간색이고, 그라니 스미스는 초록색이며, 골든 딜리셔스는 노란색입니다. 또한, 일부 사과는 두 가지 이상의 색상이 섞여 있는 경우도 있습니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 16, 'total_tokens': 118, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f70554c601', 'id': 'chatcmpl-EOwMGQTwCxdenvlY0KR5NKPP21yx1', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0ad3d-d9dc-7852-95f8-bcc2e5b2be44-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 16, 'output_tokens': 102, 'total_tokens

In [18]:
from langchain_core.load import load, loads

load_chain = load(
    loaded_chain, secrets_map= {"OPENAI_API_KEY": os.environ["OPENAI_API_KEY"]},allowed_objects="all"
)

load_chain.invoke({"fruit":"사과"})

AIMessage(content='사과의 색상은 다양합니다. 일반적으로 빨간색, 초록색, 노란색 등 여러 가지 색상이 있으며, 품종에 따라 다르게 나타납니다. 예를 들어, 레드 딜리셔스는 주로 빨간색이고, 그라니 스미스는 초록색이며, 골든 딜리셔스는 노란색입니다. 또한, 일부 사과는 두 가지 이상의 색상이 섞여 있는 경우도 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 16, 'total_tokens': 118, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f70554c601', 'id': 'chatcmpl-EOwP9cZiDK80wD6bmAKQlTyhnCUHt', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0ad40-9a07-7243-b28f-93962c06422b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 10

In [19]:
with open("fruit_chain.json", "r") as fp:
    loaded_from_json_chain = json.load(fp)
    loads_chain = load(loaded_from_json_chain, allowed_objects="all")

loads_chain.invoke({"fruit":"사과"})

AIMessage(content='사과의 색상은 다양합니다. 일반적으로 빨간색, 초록색, 노란색 등 여러 가지 색상이 있으며, 품종에 따라 다르게 나타납니다. 예를 들어, 레드 딜리셔스는 주로 빨간색이고, 그라니 스미스는 초록색이며, 골든 딜리셔스는 노란색입니다. 또한, 일부 사과는 두 가지 이상의 색상이 섞여 있는 경우도 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 16, 'total_tokens': 118, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f70554c601', 'id': 'chatcmpl-EOwR6lmDv8Ku9cK67IJ1XpwhirDvq', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0ad42-6dd7-7d12-9d2d-4ebb43fe8ed2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 10